## **Baseline Model Evaluation**

This notebook evaluates the winning solution from the [AICrowd Airborne Object Tracking Challenge](https://www.aicrowd.com/challenges/airborne-object-tracking-challenge) (2021) as the project baseline. The solution, developed by Dmytro Poplavskiy, uses a segmentation-based detection and tracking pipeline (SegTracker) with an ensemble of four deep learning models: HRNet32, HRNet48, GerNet-M with BiFPN, and DLA60.

Inference is performed across the entire custom hard dataset subset [outlined below](#dataset), with results presented both as an overall set of benchmark statistics and as performance metrics for each data subset.

**Evaluation Metrics:**
- **AFDR:** Airborne Flight Detection Rate. The primary detection metric.
- **FPPI:** False Positives Per Image. Must be ≤ 0.0005 to be ranked.
- **HFAR:** Horizontal False Alarm Rate. Must be ≤ 0.5 to be ranked.
- **EDR:** Encounter Detection Rate. Tracking-based metric across all horizon scenarios.

> ⚠️ ***Note:*** *Inference requires a CUDA-capable GPU. The `map_location` fix in `seg_tracker.py` allows CPU loading of weights for inspection, but full inference at acceptable speed requires GPU. Estimated runtime on GPU: ~2–3 FPS per flight sequence.*

In [ ]:
# Import Required Libraries.
import json
import os
import cv2
import random
import shutil
import sys
from glob import glob

import matplotlib.pyplot as plt
import pandas as pd
import torch

# Because Life, Universe & Everything!
random.seed(42)

# Set Working Directory to Project Root.
os.chdir("..") 
PROJECT_ROOT = os.getcwd()

# Add Project Root to Path.
sys.path.insert(0, PROJECT_ROOT)

# Add Core Metric Folders to System Path.
sys.path.insert(0, os.path.join(PROJECT_ROOT, "core"))

# Import Local Project Funtions.
from logging_module import LogLevel, log
from src.baseline.seg_test import SegPredictor
from src.data.download_data import main as downloadData
from notebooks.helpers.notebook_visualiser import main as visualise_results
from notebooks.helpers.notebook_helpers import get_next_run_dir, load_combined_gt, resolve_local_flights, convert_results_to_metrics_format, prepare_subset, load_subset_summary

# Confirm Project Root.
log(LogLevel.INFO, "baseline:imports", "Project Root Set: %s", PROJECT_ROOT)


---

<br/>

## **Dataset**

The custom dataset used in this notebook is a curated hard subset of the full Airborne Object Tracking (AOT) dataset, specifically designed to stress-test model performance under the most demanding real-world conditions. Flights were selected to isolate encounters that represent the greatest challenge for vision-based detection and tracking systems.

**Isolation Criteria - encounters were selected for:**
- **Small Targets:** Airborne objects occupying only a few pixels per frame.
- **Multiple Targets:** Scenes containing more than one airborne object simultaneously.
- **Poor Visibility:** Flights affected by sun glare, adverse lighting, or challenging camera angles.

Together these conditions reflect the edge cases where sense-and-avoid systems are most likely to fail, and where reliable early detection is most critical for safe autonomous flight.

### **Airborne Object Tracking (AOT) - Hard Subset Summary**

| Metric                   | Value   | Description                                                             |
|--------------------------|--------:|-------------------------------------------------------------------------|
| **Total Flights**        | **1618**| **Total number of flight sequences in the dataset**                     |
| Small Object Flights     | 662     | Flights containing at least one object with bounding box area ≤ 36.0px².|
| Below Horizon Flights    | 1053    | Flights containing at least one object at or below the horizon.         |
| Multi-Object Flights     | 627     | Flights containing more than one airborne object in at least one frame. |
| No Object Frames         | 354     | Flights containing at least one frame with no airborne object present.  |
| *Avg. Sequence Length*   | *17.77s*| *Average duration of each flight sequence.*                             |

In [ ]:
# Set Path to Desired Evaluation Dataset.
DATASET_PATH = "configs/datasets/hard_dataset_flights.txt"
DATASET_PARTS = ["part1", "part2", "part3"]

# Set Results Variables (Auto-Incrementing Run Directory).
RESULTS_DIR = "outputs/results/baseline/"
run_name = get_next_run_dir(RESULTS_DIR)

# Evaluation Filtering.
MIN_SCORE = 0.60
MIN_TRACK_LEN = 30

# Evaluation Output Folders.
EVAL_GT_DIR = os.path.join(RESULTS_DIR, run_name, "gt")
EVAL_RESULT_DIR  = os.path.join(RESULTS_DIR, run_name, "result")
EVAL_SUMMARY_DIR = os.path.join(RESULTS_DIR, run_name, "summaries")

# Print Summary Statement.
log(LogLevel.INFO, "baseline:config", "Configuration Loaded.")
print(f"  Dataset Path  : {DATASET_PATH}")
print(f"  Dataset Parts : {DATASET_PARTS}")
print(f"  Run Name      : {run_name}")
print(f"  Results Path  : {RESULTS_DIR}")
print(f"  Min Score     : {MIN_SCORE}")
print(f"  Min Track Len : {MIN_TRACK_LEN}")

In [ ]:
# Download the Selected Flight Sequences.
downloadData(DATASET_PATH)


---

<br/>

## **Data Preparation and Preprocessing**

Before inference can begin, the dataset must be resolved from the flight ID list down to what is actually available on disk, and the ground truth must be filtered to match. This cell performs two steps:

**1. Flight Resolution:** Parses the `.txt` flight ID file produced by `data_builder.py`, cross-references each entry against the local `data/<part>/Images/` directories, and builds a lookup of every flight that is both listed and downloaded. Flights in the list that have not been downloaded are logged and skipped.

**2. Ground Truth Filtering:** Loads the `groundtruth.json` file from each part's `ImageSets/` folder and merges them into a single structure, retaining only the entries that correspond to locally available flights. Entries for flights not on disk are silently skipped - this is expected, as the ground truth files reference the full AOT dataset rather than just the downloaded subset.

In [ ]:
# Find all Available Sequences from the Provided Dataset.
local_flights = resolve_local_flights(DATASET_PATH, PROJECT_ROOT)

# Add Spacing in Console Output for Readability.
print()

# Log How Many Flights were Found & are Ready for Inference.
log(LogLevel.INFO, "baseline:preprocessing", "Flights Ready for Inference: %d", len(local_flights))

# Preview the First 5 Flights to Confirm Dataset is Correct.
for name, info in list(local_flights.items())[:5]:
    # Count No. Images in the Flight Directory.
    n_imgs = len(list(info["images_dir"].glob("*.png")))

    # Print Flight Part, Name, & No. Images.
    print(f"  [{info['part']}]  {name}  ({n_imgs} images)")

# If there are More than 5 Flights, Log How Many Additional Ones Exist.
if len(local_flights) > 5:
    log(LogLevel.DEBUG, "baseline:preprocessing", " ... and %d more.", len(local_flights) - 5)

# Load & Filter Ground Truth to Include Local Flights Only. 
combined_gt = load_combined_gt(DATASET_PARTS, local_flights, PROJECT_ROOT)
log(LogLevel.INFO, "baseline:preprocessing", "Flights in Filtered Ground Truth: %d", len(combined_gt['samples']))

# Build Set of Image Filenames That Physically Exist on Disk.
all_images = {name for f in local_flights.values() for name in f["image_names"]}

# Cull GT Entities to Only Those With a Corresponding Image on Disk.
before = sum(len(flight["entities"]) for flight in combined_gt["samples"].values())

for flight_hash, flight_data in combined_gt["samples"].items():
    flight_data["entities"] = [
        e for e in flight_data["entities"]
        if e["img_name"] in all_images
    ]

after = sum(len(flight["entities"]) for flight in combined_gt["samples"].values())

# Report How Many Images were Removed. 
log(LogLevel.INFO, "baseline:preprocessing",
    "GT Culled: %d -> %d entities (%d removed for missing images).", before, after, before - after)

# Save the Filtered Ground Truth to the Evaluation Folder.
os.makedirs(EVAL_GT_DIR, exist_ok=True)
gt_out_path = os.path.join(EVAL_GT_DIR, "groundtruth.json")

# Copy the Ground Truth Values to the .json.
with open(gt_out_path, "w") as f:
    json.dump(combined_gt, f)

# Print Success Message.
log(LogLevel.INFO, "baseline:preprocessing", "Ground Truth Saved: %s", gt_out_path)


---

<br/>

## **Evaluate Baseline Model**

### **Run Baseline Inference**

**1.** Prepare evaluation directory and configure environment variables.

In [ ]:
# Create a Directory where all Evaluation Flight Data will be Stored.
EVAL_DATA_DIR = os.path.join(RESULTS_DIR, run_name, "data")
os.makedirs(EVAL_DATA_DIR, exist_ok=True)

# Link Each Local Flight's Image Folder into the Evaluation Directory.
for folder_name, info in local_flights.items():
    link_path = os.path.join(EVAL_DATA_DIR, folder_name)
    if not os.path.exists(link_path):
        os.symlink(info["images_dir"], link_path)

# Set Environment Variables that AirbornePredictor and SegPredictor Read.
os.environ["TEST_DATASET_PATH"] = EVAL_DATA_DIR
os.environ["INFERENCE_OUTPUT_PATH"] = RESULTS_DIR
os.environ["DATASET_ENV"] = run_name
os.environ["MIN_TRACK_LEN"] = str(MIN_TRACK_LEN)
os.environ["MIN_SCORE"] = str(MIN_SCORE)

# Limit Inference to Only the Specified Flights.
os.environ["PARTIAL_RUN_FLIGHTS"] = ",".join(local_flights.keys())

# Print Summary Statement.
log(LogLevel.INFO, "baseline:inference", "Environment Configured.")
print(f"  Test Data Path     : {EVAL_DATA_DIR}")
print(f"  Flights to Process : {len(local_flights)}")
print(f"  MIN_SCORE          : {MIN_SCORE}")
print(f"  MIN_TRACK_LEN      : {MIN_TRACK_LEN}")

**2.** Run segmentation predictor on all evaluation flights.

> ⚠️ ***This cells requires a CUDA GPU and will take several hours to complete.*** *Skip this cell if inference has already been run and results are present in `RESULTS_DIRS`.*

In [ ]:
# Check GPU is CUDA Enabled.
if torch.cuda.is_available():
    log(LogLevel.INFO, "baseline:device_check", "CUDA-Enabled GPU is Available.")
    print(f"  GPU Name: {torch.cuda.get_device_name(0)}.")
    print(f"  CUDA Version: {torch.version.cuda}.")
else:
      log(LogLevel.WARNING, "baseline:device_check", "No CUDA-Enabled GPU Found, Using CPU Instead.")

In [ ]:
# Initialise Predictor (Reads Env Variables).
predictor = SegPredictor()

# Run Inference Over all Flights.
predictor.run()

# Print Summary Statement.
log(LogLevel.INFO, "baseline:inference", "Inference Complete.")
print(f"  Raw Results Saved to: {os.path.join(RESULTS_DIR, run_name)}\n")

# Set Results Directories. 
RAW_RESULTS_PATH = os.path.join(RESULTS_DIR, run_name, "result.json")
METRICS_RESULTS_PATH = os.path.join(EVAL_RESULT_DIR, "result.json")

# Ensure Output Directory Exists.
os.makedirs(EVAL_RESULT_DIR, exist_ok=True)

# Copy the Results File.
convert_results_to_metrics_format(RAW_RESULTS_PATH, METRICS_RESULTS_PATH)

# Print Summary Statement.
log(LogLevel.INFO, "baseline:inference", "Results Copied to Metrics Directory.")
print(f"  Metrics-Ready Results Stored at : {METRICS_RESULTS_PATH}")

### **Run Metrics Evaluation**

Run the official AICrowd evaluation metrics against the ground truth for each dataset part, saving per-part summaries to `SUMMARIES_DIR`.

In [ ]:
# Locate the Competition Metrics Script.
metrics_script = os.path.join(PROJECT_ROOT, "core/metrics/run_airborne_metrics.py")

# Call the Competition Metrics Script.
!python3 {metrics_script} \
    --dataset-folder   {EVAL_GT_DIR} \
    --results-folder   {EVAL_RESULT_DIR} \
    --summaries-folder {EVAL_SUMMARY_DIR} \
    --min-score        {MIN_SCORE} \
    --min-track-len    {MIN_TRACK_LEN}

# Print Message on Completion. 
log(LogLevel.INFO, "baseline:evaluation", "Metrics Complete.")

### **Baseline Benchmark Summary**

Display the aggregated statistics across all three parts of the dataset in order to provide an overall baseline performance for the full hard dataset. Additionally, a random sample of results is generated using both the ground truth bounding boxes and the predicted ones, enabling a visual analysis of the model’s performance.

In [ ]:
# Find the Summary JSON in the Summaries Directory.
summary_files = glob(os.path.join(EVAL_SUMMARY_DIR, "*.json"))

# Check that the Summary Files Exist.
if not summary_files:
    log(LogLevel.ERROR, "baseline:summary", "No Summary Files Found! Run Inference First.")
else:
    # Load the Summary JSON.
    with open(summary_files[0]) as f:
        summary = json.load(f)

    # Extract Key Metrics from Summary JSON.
    score         = summary["min_det_score"]
    fppi          = summary["fppi"]
    hfar          = summary["far"]
    afdr          = summary["fl_dr_in_range"]
    det_aircraft  = summary["det_aircraft_in_range"]
    tot_aircraft  = summary["tot_aircraft_in_range"]
    edr_all       = summary["Detection"]["Encounters"]["300"]["All"]["dr"]
    edr_above     = summary["Detection"]["Encounters"]["300"]["Above Horizon"]["dr"]
    edr_below     = summary["Detection"]["Encounters"]["300"]["Below Horizon"]["dr"]

    # Rank Metrics Against Thresholds.
    fppi_status = "PASS (≤ 0.0005)" if fppi  <= 0.0005 else f"FAIL (> 0.0005)"
    hfar_status = "PASS (≤ 0.5)"    if hfar  <= 0.5    else f"FAIL (> 0.0005)"

    # Build Summary DataFrame.
    summary_df = pd.DataFrame([
        {"Metric": "Detection Score",                       "Value": f"{score:.4f}",       "Status": ""},
        {"Metric": "AFDR (Airborne Flight Detection Rate)", "Value": f"{afdr:.4f}",        "Status": ""},
        {"Metric": "FPPI (False Positives Per Image)",      "Value": f"{fppi:.6f}",        "Status": fppi_status},
        {"Metric": "HFAR (Horizontal False Alarm Rate)",    "Value": f"{hfar:.4f}",        "Status": hfar_status},
        {"Metric": "EDR - All Scenarios",                   "Value": f"{edr_all:.4f}",     "Status": ""},
        {"Metric": "EDR - Above Horizon",                   "Value": f"{edr_above:.4f}",   "Status": ""},
        {"Metric": "EDR - Below Horizon",                   "Value": f"{edr_below:.4f}",   "Status": ""},
        {"Metric": "Total Aircraft Detected",               "Value": f"{det_aircraft}",    "Status": ""},
        {"Metric": "Total Aircraft in Dataset",             "Value": f"{tot_aircraft}",    "Status": ""},
    ])

    # Print Summary to Console.
    print("\n" + "=" * 60)
    print(" BASELINE MODEL - FULL HARD DATASET BENCHMARK SUMMARY")
    print(" Model: Dmytro Poplavskiy Winning Solution (AICrowd 2021)")
    print("=" * 60)
    display(summary_df.set_index("Metric"))

    # Save Aggregated Summary to CSV.
    output_path = os.path.join(EVAL_SUMMARY_DIR, "baseline_benchmark_summary.csv")
    summary_df.to_csv(output_path, index=False)
    print()
    log(LogLevel.INFO, "baseline:summary", "Summary Saved to: %s", output_path)

In [ ]:
# Set Visualisations Folder Path. 
VISUALISER_DIR = os.path.join(RESULTS_DIR, run_name, "visualisations")

# Make & Save the Model Prediction Overlay Pictures.
visualise_results(
  results_csv = os.path.join(EVAL_RESULT_DIR, "result.csv"),
  groundtruth_csv = os.path.join(EVAL_GT_DIR, "groundtruth.csv"),
  images_dir = EVAL_DATA_DIR,
  output_dir = VISUALISER_DIR
)

# Get Image Files (Sorted for Consistency).
image_files = sorted([f for f in os.listdir(VISUALISER_DIR) if f.endswith(".png")])

# Limit the Number of Images to 10 to Fit a Grid of 2 Columns by 5 Rows.
image_files = image_files[:10]
cols = 2
rows = 5

# Create a Matplotlib Figure with the Specified Grid Size & Figure Dimensions.
fig, axes = plt.subplots(rows, cols, figsize=(10, 20))

# Iterate Over each Subplot Axis & Corresponding Image Index.
for i, ax in enumerate(axes.flatten()):
    if i < len(image_files):
        # Build the Full Image Path.
        img_path = os.path.join(VISUALISER_DIR, image_files[i])
        
        # Read + Convert BGR --> RGB.
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        # Display the Image in the Current Subplot.
        ax.imshow(img)

        # Clean Up the Image Filename to Show a Simpler Title.
        clean_name = image_files[i].split('_', 2)[-1]
        ax.set_title(clean_name, fontsize=8)

        # Hide the Axis Ticks & Labels for a Cleaner Look.
        ax.axis("off")
    else:
        # If there are Fewer Images than Grid Spaces, Hide Unused Axes.
        ax.axis("off")

# Adjust Spacing Between Subplots to Prevent Overlap.
plt.tight_layout()

# Display the Grid of Images.
plt.show()


---

<br/>

## **Hard Dataset Results**

The table below reports baseline performance broken down by scenario category across the custom
hard dataset (1,618 flights). Flights were selected from the full AOT dataset using configurable
filter criteria to isolate the most challenging sequences, providing a more targeted benchmark
than the full competition dataset.

| Scenario                     | Flights | FPPI | AFDR | HFAR | EDR |
|------------------------------|---------|------|------|------|-----|
| **Overall (All Criteria)**   |**1,618**| —    | —    | —    | —   |
| Small Object (bbox ≤ 36 px²) | 662     | —    | —    | —    | —   |
| Below Horizon                | 1,053   | —    | —    | —    | —   |
| Multi-Object                 | 627     | —    | —    | —    | —   |
| No Visible Object            | 354     | —    | —    | —    | —   |

*Results evaluated at a minimum detection confidence of 0.985 and a minimum track length of 30 frames (≈ 3 seconds at 10 fps).*

In [ ]:
# List of Text Files that Define Challenging Subsets of Flight Data.
_subsets = [
    "configs/datasets/hard_subsets/small_object_flights.txt",   # Flights with Small Objects.
    "configs/datasets/hard_subsets/horizon_flights.txt",        # Flights with Objects Below Horizon.
    "configs/datasets/hard_subsets/multi_object_flights.txt",   # Flights with Multiple Objects.
    "configs/datasets/hard_subsets/no_object_flights.txt",      # Flights with No Objects.
]

# Iterate Over Each Subset File to Evaluate its Corresponding Dataset.
for _subset in _subsets:
    # Prepare Paths & Names for the Subset.
    subset_name, subset_result_dir, subset_gt_dir, subset_summary_dir = prepare_subset(
        subset_txt   = _subset,                             # Path to the Subset Definition File.
        run_dir      = os.path.join(RESULTS_DIR, run_name), # Output Directory for Results.
        run_gt_dir   = EVAL_GT_DIR,                         # Directory Containing Ground Truth Annotations.
        project_root = PROJECT_ROOT,                        # Root Directory of the Project.
    )

    # Run Evaluation Metrics for this Subset Using an External Script.
    !python3 {metrics_script} \
        --dataset-folder   {subset_gt_dir} \
        --results-folder   {subset_result_dir} \
        --summaries-folder {subset_summary_dir} \
        --min-score        {MIN_SCORE} \
        --min-track-len    {MIN_TRACK_LEN}

    # Load the Evaluation Summary for this Subset.
    summary_df = load_subset_summary(subset_summary_dir, subset_name)

    # If the Summary was Successfully Loaded, Display it Nicely.
    if summary_df is not None:
        print("\n" + "=" * 60)
        print(f" SUBSET EVALUATION SUMMARY — {subset_name.upper()}")
        print("=" * 60)

        # Display the Results with Colours.
        display(summary_df.set_index("Metric").style
            .applymap(lambda v: "color: green" if v and "PASS" in str(v) else
                                "color: red"   if v and "FAIL" in str(v) else "",
                                subset=["Status"])
        )

        # Add Spacing Between Subsets.
        print()

---